# Poisson & Gamma GLMs: The Regulatory Baseline

**Author:** Andres Gonzalez Ortega | **Date:** March 2026

---

## Why GLMs Are the Industry Standard

Generalized Linear Models have been the backbone of insurance pricing since the 1990s. Their advantages:

1. **Interpretability**: Every coefficient has a direct actuarial interpretation as a *relativity* (multiplicative factor)
2. **Regulatory acceptance**: Regulators (CNSF in Mexico, ACPR in France) understand and trust GLMs
3. **Statistical rigor**: Proper handling of non-Normal distributions via the exponential family
4. **Decomposition**: Natural fit for the frequency-severity framework

### The Mathematical Framework

**Frequency (Poisson GLM):**
- Y ~ Poisson(mu * Exposure)
- log(mu) = beta_0 + beta_1*x_1 + ... + beta_p*x_p + offset(log(Exposure))
- Each exp(beta_j) is a *relativity*: the multiplicative effect on claim frequency

**Severity (Gamma GLM):**
- Y | Y > 0 ~ Gamma(mu, alpha)
- log(mu) = beta_0 + beta_1*x_1 + ... + beta_p*x_p
- No exposure offset needed (severity is not exposure-dependent)

> *En Mexico, la Circular Unica de Seguros de la CNSF requiere que las tarifas sean "tecnica y actuarialmente sustentadas". Los GLMs cumplen este requisito perfectamente al proporcionar coeficientes interpretables y pruebas de significancia estadistica.*

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parents[0] if Path().resolve().name == 'notebooks' else Path().resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
sys.path.insert(0, str(PROJECT_ROOT.parent.parent))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import seaborn as sns
import matplotlib.pyplot as plt

pio.templates.default = "plotly_white"
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

import statsmodels.api as sm
from scipy import stats

## 1. Data Preparation

In [ ]:
from scripts.utils.data_loader import load_merged, split_data
from feature_engineering import create_features, get_glm_feature_names

# Load and split
df = load_merged()
train, val, test = split_data(df)

# Create GLM features (one-hot encoded)
train_glm = create_features(train, for_glm=True)
val_glm = create_features(val, for_glm=True)
test_glm = create_features(test, for_glm=True)

feature_names = get_glm_feature_names(train_glm)
print(f"Number of GLM features: {len(feature_names)}")
print(f"Feature names: {feature_names[:15]}...")
print(f"\nTrain: {len(train_glm):,} | Val: {len(val_glm):,} | Test: {len(test_glm):,}")

## 2. Train Poisson GLM for Frequency

In [ ]:
from glm_models import train_poisson_glm, predict_glm, extract_coefficients, glm_summary_stats

poisson_result = train_poisson_glm(
    X_train=train_glm,
    y_train=train_glm['ClaimNb'],
    exposure_train=train_glm['Exposure'],
    feature_names=feature_names,
)

## 3. Model Summary

In [ ]:
print(poisson_result.summary())

## 4. Coefficient Table with Relativities

In [ ]:
coefficients = extract_coefficients(poisson_result, top_n=40)
coef_df = pd.DataFrame(coefficients)

# Add relativity interpretation
coef_df['relativity_pct'] = (coef_df['relativity'] - 1) * 100

display(coef_df[['feature', 'coefficient', 'relativity', 'relativity_pct', 'p_value', 'significant']]
        .head(30))

### Understanding Relativities

A **relativity** of 1.25 means the claim frequency is 25% higher than the reference level.
A relativity of 0.80 means 20% lower. These are multiplicative factors in the pricing formula:

**Predicted frequency = base_rate x relativity_1 x relativity_2 x ... x relativity_p**

This is exactly how actuaries communicate rates to underwriters: "A young driver in Paris with a high BonusMalus gets a 3.2x surcharge relative to the base rate."

## 5. Relativity Visualization

In [ ]:
# Plot top relativities (sorted by absolute deviation from 1.0)
coef_plot = coef_df[coef_df['feature'] != 'const'].copy()
coef_plot['abs_effect'] = abs(coef_plot['relativity'] - 1)
coef_plot = coef_plot.nlargest(20, 'abs_effect')

fig = px.bar(coef_plot, y='feature', x='relativity_pct',
             orientation='h',
             title='Top 20 GLM Relativities (% Change from Base)',
             labels={'relativity_pct': 'Relativity (% change)', 'feature': 'Feature'},
             color='relativity_pct',
             color_continuous_scale='RdBu_r',
             color_continuous_midpoint=0)
fig.update_layout(height=600, yaxis={'categoryorder': 'total ascending'})
fig.show()

### So What?
The relativities tell the actuarial story:
- **BonusMalus bins** dominate: the highest BM bin (151+) has a relativity >2x. This is expected -- it is the most informative feature.
- **Area F** (Paris/urban) shows elevated frequency vs the reference area.
- **Young driver bins** (18-22, 23-26) have high relativities, confirming the U-shaped curve from EDA.
- **Vehicle brand** effects are modest but statistically significant.

> *Para la CNSF, esta tabla de relatividades ES la tarifa tecnica. Cada factor multiplicativo se aplica al prima base para obtener la prima de riesgo individual.*

## 6. Residual Diagnostics

In [ ]:
# Deviance residuals
fitted_values = poisson_result.predict(
    sm.add_constant(train_glm[feature_names].astype(float)),
    offset=np.log(train_glm['Exposure'].clip(lower=1e-9))
)
resid_dev = poisson_result.resid_deviance

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs fitted
axes[0].scatter(fitted_values[:5000], resid_dev[:5000], alpha=0.2, s=5, c='#2E86AB')
axes[0].axhline(y=0, color='red', linestyle='--')
axes[0].set_xlabel('Fitted Values')
axes[0].set_ylabel('Deviance Residuals')
axes[0].set_title('Deviance Residuals vs Fitted Values')

# QQ plot
stats.probplot(resid_dev[:10000], dist="norm", plot=axes[1])
axes[1].set_title('QQ Plot of Deviance Residuals')
axes[1].get_lines()[0].set_markersize(2)

plt.tight_layout()
plt.show()

### So What?
Poisson GLM residuals will never look "perfectly Normal" due to the discrete, zero-inflated nature of claim counts.
What we look for:
- No systematic pattern in residuals vs fitted (curved bands are expected with count data)
- No extreme outliers suggesting data errors
- The QQ plot departs from normality in the tails -- this is typical for Poisson data and not a model failure

The diagnostics confirm the model is behaving as expected for count data.

## 7. Gamma GLM for Severity

In [ ]:
from glm_models import train_gamma_glm

gamma_result = train_gamma_glm(
    X_train=train_glm,
    y_train=train_glm['AvgClaimAmount'],
    feature_names=feature_names,
)

print("\n--- Gamma GLM Summary ---")
gamma_stats = glm_summary_stats(gamma_result)
for k, v in gamma_stats.items():
    print(f"  {k}: {v}")

## 8. Severity Coefficient Interpretation

In [ ]:
sev_coefficients = extract_coefficients(gamma_result, top_n=20)
sev_df = pd.DataFrame(sev_coefficients)
sev_df['relativity_pct'] = (sev_df['relativity'] - 1) * 100

fig = px.bar(sev_df[sev_df['feature'] != 'const'].nlargest(15, 'relativity_pct'),
             y='feature', x='relativity_pct',
             orientation='h',
             title='Gamma GLM: Severity Relativities (% Change)',
             labels={'relativity_pct': 'Relativity (% change)', 'feature': 'Feature'},
             color_discrete_sequence=['#D64933'])
fig.update_layout(height=500, yaxis={'categoryorder': 'total ascending'})
fig.show()

### So What?
Severity relativities tell a different story than frequency:
- **VehPower** matters more for severity (powerful vehicles = more expensive crashes)
- **BonusMalus** matters less for severity (bad drivers crash more often, but not necessarily more expensively)
- **Area** effects may reverse: urban claims can be less severe (lower speeds) even though more frequent

This validates the frequency-severity decomposition: the drivers of "how often" and "how much" are different.

## 9. Test Set Evaluation

In [ ]:
from scripts.utils.evaluation import (
    compute_all_metrics, lorenz_curve, calibration_data, lift_chart
)

# Predict on test set
y_test = test_glm['ClaimNb']
exposure_test = test_glm['Exposure']
freq_pred_glm = predict_glm(poisson_result, test_glm, feature_names, exposure=exposure_test)

# Compute metrics
metrics = compute_all_metrics(y_test, freq_pred_glm, sample_weight=exposure_test, family='poisson')
print("Poisson GLM -- Test Set Metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v}")

## 10. Calibration Plot

In [ ]:
cal = calibration_data(y_test, freq_pred_glm, sample_weight=exposure_test)
cal_df = pd.DataFrame(cal['bins'])

fig = go.Figure()
fig.add_trace(go.Scatter(x=cal_df['avg_predicted'], y=cal_df['avg_actual'],
                         mode='markers+lines', name='Model',
                         marker=dict(size=10, color='#2E86AB')))
# Add perfect calibration line
min_val = min(cal_df['avg_predicted'].min(), cal_df['avg_actual'].min())
max_val = max(cal_df['avg_predicted'].max(), cal_df['avg_actual'].max())
fig.add_trace(go.Scatter(x=[min_val, max_val], y=[min_val, max_val],
                         mode='lines', name='Perfect Calibration',
                         line=dict(dash='dash', color='gray')))

fig.update_layout(title='Calibration Plot: Predicted vs Actual Claim Frequency (by Decile)',
                  xaxis_title='Average Predicted Frequency',
                  yaxis_title='Average Actual Frequency')
fig.show()

## 11. Lorenz Curve

In [ ]:
lorenz = lorenz_curve(y_test, freq_pred_glm, sample_weight=exposure_test)

fig = go.Figure()
fig.add_trace(go.Scatter(x=lorenz['fraction_policies'], y=lorenz['fraction_claims'],
                         mode='lines', name=f'Poisson GLM (Gini={metrics["gini"]:.4f})',
                         line=dict(color='#2E86AB', width=2)))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1],
                         mode='lines', name='Random Model',
                         line=dict(dash='dash', color='gray')))

fig.update_layout(title='Lorenz Curve -- Poisson GLM Risk Discrimination',
                  xaxis_title='Fraction of Policies (ordered by predicted risk)',
                  yaxis_title='Fraction of Claims',
                  legend=dict(x=0.05, y=0.95))
fig.show()

### So What?
The Lorenz curve shows how well the model separates high-risk from low-risk policyholders.
A perfect model would concentrate all claims among the policies it rates highest.
The Gini coefficient quantifies this: higher = better discrimination.

The GLM achieves a solid baseline Gini -- but can ML models do better? That is the question for the next notebook.

---

## Summary

| Metric | Value |
|--------|-------|
| Model | Poisson GLM (log link, Exposure offset) |
| Features | One-hot encoded categoricals + binned numerics |
| Interpretability | Full -- every coefficient is a relativity |
| Regulatory readiness | High -- standard actuarial methodology |

**What the GLM cannot capture:**
- Non-linear continuous effects (must be manually binned)
- Feature interactions (unless explicitly added)
- Complex patterns in high-cardinality features

**Next notebook: 03_gbm_models.ipynb** -- We train XGBoost and LightGBM to see where ML beats the GLM.